# siteHopper v3 — SQLite graph-store refactor

This notebook refactors the original `siteHopper-2.ipynb` storage layer to support efficient **graph-native caching**:

- ✅ Persistent SQLite DB (file-backed)
- ✅ Normalized schema: `pages` + `edges` table (no comma-separated hyperlinks)
- ✅ Canonical page keys (single representation across fetch/parse/store)
- ✅ Freshness metadata: `content_sha256`, `status_code`, `fetched_at`
- ✅ Smart batching for crawls (transaction + `executemany`)
- ✅ Benchmarks: runtime + storage footprint
- ✅ Visualizations (static + interactive Plotly)

**Note:** The benchmark/tests use a deterministic `FakeWikipediaScraper` so the notebook runs without network access.
You can swap in the real `WikipediaScraper` when running locally with internet.


In [ ]:
%pip install plotly

In [ ]:
import os
import re
import time
import math
import json
import sqlite3
import hashlib
import random
import datetime
from dataclasses import dataclass
from typing import Iterable, Optional, List, Dict, Tuple, Any

import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt

from urllib.parse import urlparse, unquote

# Plotly is used for a higher-fidelity interactive graph export
import plotly.graph_objects as go
import plotly.io as pio


## 1) Utilities — canonical keys, hashing, timestamps

A *canonical page key* is the normalized identifier for a Wikipedia entity.
Examples (all map to the same key):

- `"/wiki/Python_(programming_language)"`
- `"Python (programming language)"`
- `"https://en.wikipedia.org/wiki/Python_(programming_language)#History"`

Canonicalization gives you one consistent representation across:
- crawler → store
- store → graph search
- graph search → explanation path / visualization


In [ ]:
WIKI_BASE_URL = "https://en.wikipedia.org"
WIKI_PATH_PREFIX = "/wiki/"

def utc_now_iso() -> str:
    """UTC timestamp string (seconds precision)"""
    return datetime.datetime.utcnow().replace(microsecond=0).isoformat() + "Z"

def sha256_text(text: str) -> str:
    return hashlib.sha256(text.encode("utf-8", errors="ignore")).hexdigest()

def canonicalize_page_key(page: str) -> str:
    """Return a canonical Wikipedia page key (title) from many possible inputs.

    Accepts:
      - '/wiki/Title'
      - 'Title' (spaces ok)
      - full URL 'https://en.wikipedia.org/wiki/Title#fragment'
    Returns:
      - 'Title' with spaces replaced by underscores, no fragments/queries.
    """
    if page is None:
        raise ValueError("page cannot be None")
    s = str(page).strip()
    if not s:
        raise ValueError("page cannot be empty")

    # If full URL, extract path
    if s.startswith("http://") or s.startswith("https://"):
        path = urlparse(s).path
    else:
        path = s

    # Strip query/fragment
    path = path.split("#", 1)[0].split("?", 1)[0]

    # Common forms
    if path.startswith(WIKI_PATH_PREFIX):
        key = path[len(WIKI_PATH_PREFIX):]
    else:
        key = path

    key = unquote(key)
    key = key.replace(" ", "_")

    # A last sanity cleanup: remove accidental leading/trailing slashes
    key = key.strip("/")

    return key

def wiki_relpath(page_key: str) -> str:
    return f"{WIKI_PATH_PREFIX}{page_key}"

def wiki_url(page_key: str, base_url: str = WIKI_BASE_URL) -> str:
    return f"{base_url}{WIKI_PATH_PREFIX}{page_key}"


## 2) SQLite Property-Graph Store

We model Wikipedia as a **directed property graph**:

- `pages` table = nodes
- `edges` table = directed edges (`WIKI_LINK` by default)

Key design points:
- `edges` stores integer page IDs (compact + fast joins)
- `edges` is declared `WITHOUT ROWID` with a composite primary key for compactness and performance
- Batching uses **one transaction per flush** + `executemany` to reduce commit overhead


In [ ]:
@dataclass(frozen=True)
class PageRecord:
    page: str
    url: str
    content: Optional[str] = None
    status_code: Optional[int] = None
    fetched_at: Optional[str] = None

@dataclass(frozen=True)
class EdgeRecord:
    src: str
    dst: str
    rel: str = "WIKI_LINK"
    weight: Optional[float] = None
    anchor_text: Optional[str] = None
    discovered_at: Optional[str] = None


class WikiSqliteStore:
    """Efficient SQLite-backed cache + graph store."""

    def __init__(self, db_path: str = "wiki_cache.db", base_url: str = WIKI_BASE_URL):
        self.db_path = db_path
        self.base_url = base_url
        self.conn = sqlite3.connect(db_path)
        self.conn.row_factory = sqlite3.Row

        # Performance-oriented pragmas (safe defaults for a local cache)
        self.conn.execute("PRAGMA journal_mode=WAL;")
        self.conn.execute("PRAGMA synchronous=NORMAL;")
        self.conn.execute("PRAGMA foreign_keys=ON;")
        self.conn.execute("PRAGMA temp_store=MEMORY;")

        # In-process cache to reduce repeated SELECTs during crawls
        self._id_cache: Dict[str, int] = {}

        self._init_schema()

    def close(self):
        self.conn.close()

    def _init_schema(self):
        self.conn.executescript("""
        CREATE TABLE IF NOT EXISTS pages (
            page_id       INTEGER PRIMARY KEY,
            page          TEXT NOT NULL UNIQUE,      -- canonical key
            url           TEXT NOT NULL,
            content       TEXT,
            content_sha256 TEXT,
            status_code   INTEGER,
            fetched_at    TEXT
        );

        CREATE TABLE IF NOT EXISTS edges (
            src_page_id   INTEGER NOT NULL,
            dst_page_id   INTEGER NOT NULL,
            rel           TEXT NOT NULL DEFAULT 'WIKI_LINK',
            weight        REAL,
            anchor_text   TEXT,
            discovered_at TEXT,
            PRIMARY KEY (src_page_id, dst_page_id, rel),
            FOREIGN KEY (src_page_id) REFERENCES pages(page_id),
            FOREIGN KEY (dst_page_id) REFERENCES pages(page_id)
        ) WITHOUT ROWID;

        -- For incoming-link queries (optional but cheap)
        CREATE INDEX IF NOT EXISTS idx_edges_dst ON edges(dst_page_id);
        """)
        self.conn.commit()

    # ---------- basic stats ----------
    def db_component_sizes(self) -> Dict[str, int]:
        """Return on-disk sizes of SQLite components (db + wal + shm).

        Useful for empirical storage benchmarks when using WAL mode.
        """
        base = self.db_path
        parts = {
            "db": base,
            "wal": base + "-wal",
            "shm": base + "-shm",
        }
        sizes: Dict[str, int] = {}
        for name, path in parts.items():
            sizes[name] = os.path.getsize(path) if os.path.exists(path) else 0
        return sizes

    def disk_usage_bytes(self) -> int:
        sizes = self.db_component_sizes()
        return int(sum(sizes.values()))


    def counts(self) -> Dict[str, int]:
        cur = self.conn.cursor()
        cur.execute("SELECT COUNT(*) AS n FROM pages")
        pages_n = int(cur.fetchone()["n"])
        cur.execute("SELECT COUNT(*) AS n FROM edges")
        edges_n = int(cur.fetchone()["n"])
        return {"pages": pages_n, "edges": edges_n}

    # ---------- page id helpers ----------
    def _select_page_ids(self, keys: List[str]) -> Dict[str, int]:
        """Fetch page_id for each key (chunked to avoid SQLite parameter limits)."""
        out: Dict[str, int] = {}
        keys = [k for k in keys if k]
        if not keys:
            return out

        # SQLite default variable limit is often 999; keep headroom.
        CHUNK = 900
        cur = self.conn.cursor()
        for i in range(0, len(keys), CHUNK):
            chunk = keys[i:i+CHUNK]
            qs = ",".join(["?"] * len(chunk))
            cur.execute(f"SELECT page, page_id FROM pages WHERE page IN ({qs})", chunk)
            for row in cur.fetchall():
                out[row["page"]] = int(row["page_id"])
        return out

    def _ensure_pages_exist(self, keys: Iterable[str]):
        """Insert placeholder page rows for any unknown keys."""
        keys = [canonicalize_page_key(k) for k in keys]
        keys = list(dict.fromkeys(keys))  # stable dedup

        # fast path: if many already cached
        missing = [k for k in keys if k not in self._id_cache]
        if not missing:
            return

        # Insert placeholders (url only). Content comes later via upsert.
        cur = self.conn.cursor()
        cur.executemany(
            "INSERT OR IGNORE INTO pages(page, url) VALUES(?, ?)",
            [(k, wiki_url(k, self.base_url)) for k in missing]
        )

        # Load ids into cache
        ids = self._select_page_ids(missing)
        self._id_cache.update(ids)

    def get_page(self, key: str) -> Optional[sqlite3.Row]:
        key = canonicalize_page_key(key)
        cur = self.conn.cursor()
        cur.execute(
            """SELECT page_id, page, url, content, content_sha256, status_code, fetched_at
               FROM pages WHERE page=?""",
            (key,)
        )
        return cur.fetchone()

    def neighbors(self, key: str, limit: Optional[int] = None) -> List[str]:
        key = canonicalize_page_key(key)

        # Resolve src id (cache → db)
        src_id = self._id_cache.get(key)
        if src_id is None:
            row = self.get_page(key)
            if row is None:
                return []
            src_id = int(row["page_id"])
            self._id_cache[key] = src_id

        cur = self.conn.cursor()
        sql = """SELECT p.page
                 FROM edges e
                 JOIN pages p ON p.page_id = e.dst_page_id
                 WHERE e.src_page_id = ?
                 ORDER BY e.dst_page_id"""
        params: Tuple[Any, ...]
        if limit is not None:
            sql += " LIMIT ?"
            params = (src_id, int(limit))
        else:
            params = (src_id,)
        cur.execute(sql, params)
        return [r["page"] for r in cur.fetchall()]

    # ---------- writes ----------
    def bulk_write(self, pages: List[PageRecord], edges: List[EdgeRecord]):
        """Write pages and edges in a single transaction (fast path for crawls)."""
        # Canonicalize and normalize inputs
        norm_pages: List[PageRecord] = []
        for p in pages:
            key = canonicalize_page_key(p.page)
            url = p.url or wiki_url(key, self.base_url)
            fetched_at = p.fetched_at or (utc_now_iso() if p.content is not None else None)
            norm_pages.append(PageRecord(
                page=key,
                url=url,
                content=p.content,
                status_code=p.status_code,
                fetched_at=fetched_at
            ))

        norm_edges: List[EdgeRecord] = []
        for e in edges:
            src = canonicalize_page_key(e.src)
            dst = canonicalize_page_key(e.dst)
            norm_edges.append(EdgeRecord(
                src=src,
                dst=dst,
                rel=e.rel or "WIKI_LINK",
                weight=e.weight,
                anchor_text=e.anchor_text,
                discovered_at=e.discovered_at or utc_now_iso()
            ))

        # Collect all keys we must ensure exist
        all_keys: List[str] = []
        all_keys.extend([p.page for p in norm_pages])
        for e in norm_edges:
            all_keys.append(e.src)
            all_keys.append(e.dst)
        all_keys = list(dict.fromkeys(all_keys))

        cur = self.conn.cursor()
        cur.execute("BEGIN")
        try:
            # 1) Ensure placeholders for everything (url only)
            cur.executemany(
                "INSERT OR IGNORE INTO pages(page, url) VALUES(?, ?)",
                [(k, wiki_url(k, self.base_url)) for k in all_keys]
            )

            # 2) Upsert content-bearing pages
            upsert_rows = []
            for p in norm_pages:
                if p.content is None:
                    continue
                h = sha256_text(p.content)
                upsert_rows.append((p.page, p.url, p.content, h, p.status_code, p.fetched_at))

            cur.executemany(
                """INSERT INTO pages(page, url, content, content_sha256, status_code, fetched_at)
                   VALUES(?,?,?,?,?,?)
                   ON CONFLICT(page) DO UPDATE SET
                       url=excluded.url,
                       content=excluded.content,
                       content_sha256=excluded.content_sha256,
                       status_code=excluded.status_code,
                       fetched_at=excluded.fetched_at
                """,
                upsert_rows
            )

            # 3) Load ids for keys touched in this batch
            ids = self._select_page_ids(all_keys)
            self._id_cache.update(ids)

            # 4) Insert edges by ids
            edge_rows = []
            for e in norm_edges:
                src_id = ids.get(e.src)
                dst_id = ids.get(e.dst)
                if src_id is None or dst_id is None:
                    continue
                edge_rows.append((src_id, dst_id, e.rel, e.weight, e.anchor_text, e.discovered_at))

            # Dedup edges within the batch to reduce insert work
            if edge_rows:
                edge_rows = list(dict.fromkeys(edge_rows))

            cur.executemany(
                """INSERT OR IGNORE INTO edges(
                       src_page_id, dst_page_id, rel, weight, anchor_text, discovered_at
                   ) VALUES(?,?,?,?,?,?)""",
                edge_rows
            )

            self.conn.commit()
        except Exception:
            self.conn.rollback()
            raise

    def upsert_page(self, page: PageRecord):
        self.bulk_write([page], [])

    def insert_edges(self, edges: List[EdgeRecord]):
        self.bulk_write([], edges)


## 3) Smart batching for crawls

A crawl loop typically performs:
1. fetch page
2. store page content
3. store edges

Committing after each page is slow. This `CrawlBatchWriter` accumulates writes and flushes in chunks:
- One SQLite transaction per flush
- Uses `executemany` internally via `store.bulk_write`


In [ ]:
class CrawlBatchWriter:
    def __init__(self, store: WikiSqliteStore, batch_pages: int = 50, batch_edges: int = 10_000):
        self.store = store
        self.batch_pages = int(batch_pages)
        self.batch_edges = int(batch_edges)
        self._pages: List[PageRecord] = []
        self._edges: List[EdgeRecord] = []

    def add_page(self, page: PageRecord):
        self._pages.append(page)
        self._maybe_flush()

    def add_edges(self, edges: Iterable[EdgeRecord]):
        self._edges.extend(list(edges))
        self._maybe_flush()

    def _maybe_flush(self):
        if len(self._pages) >= self.batch_pages or len(self._edges) >= self.batch_edges:
            self.flush()

    def flush(self):
        if not self._pages and not self._edges:
            return
        self.store.bulk_write(self._pages, self._edges)
        self._pages.clear()
        self._edges.clear()

    def close(self):
        self.flush()

    def __enter__(self):
        return self

    def __exit__(self, exc_type, exc, tb):
        self.close()


## 4) Scrapers

### Real `WikipediaScraper` (network)
Kept for your real runs.

### `FakeWikipediaScraper` (offline)
Used for tests and benchmarks in this notebook so results are reproducible and don’t require internet.


In [ ]:
import requests
from bs4 import BeautifulSoup

class WikipediaScraper:
    """Network scraper: fetches Wikipedia page text + outlinks."""
    def __init__(self, base_url: str = WIKI_BASE_URL, timeout: int = 15):
        self.base_url = base_url
        self.timeout = timeout
        self.headers = {
            "User-Agent": "Mozilla/5.0 (siteHopper research crawler; +https://example.org)"
        }

    def _clean_wiki_hrefs(self, hrefs: Iterable[str]) -> List[str]:
        out = []
        for href in hrefs:
            if not href:
                continue
            if not href.startswith("/wiki/"):
                continue
            href = href.split("#", 1)[0].split("?", 1)[0]
            # Drop non-article namespaces (File:, Category:, Help:, etc.)
            if ":" in href[len("/wiki/"):]:
                continue
            key = canonicalize_page_key(href)
            out.append(key)
        # stable dedup
        return list(dict.fromkeys(out))

    def fetch_page(self, page: str) -> Tuple[str, List[str], int]:
        key = canonicalize_page_key(page)
        url = wiki_url(key, self.base_url)
        resp = requests.get(url, headers=self.headers, timeout=self.timeout)
        status = int(resp.status_code)

        if status != 200:
            return "", [], status

        soup = BeautifulSoup(resp.content, "html.parser")

        # Extract outlinks
        hrefs = [a.get("href") for a in soup.find_all("a", href=True)]
        outlinks = self._clean_wiki_hrefs(hrefs)

        # Strip noisy elements for content extraction
        for element in soup(["table", "ul", "ol", "dl", "sup", "span"]):
            element.decompose()

        body = soup.find("div", {"id": "bodyContent"})
        if body:
            paragraphs = body.find_all("p")
            content = " ".join(p.get_text(" ", strip=True) for p in paragraphs)
        else:
            content = ""

        return content, outlinks, status


class FakeWikipediaScraper:
    """Deterministic offline generator for content + outlinks.

    Given a key, it produces:
      - content: pseudo-random text
      - outlinks: deterministic list of other page keys
    """
    def __init__(self, n_pages: int = 5000, out_degree: int = 12, seed: int = 123):
        self.n_pages = int(n_pages)
        self.out_degree = int(out_degree)
        self.seed = int(seed)
        self.vocab = [f"w{i}" for i in range(1000)]

    def _rng_for_key(self, key: str) -> random.Random:
        h = int(hashlib.sha256((key + str(self.seed)).encode("utf-8")).hexdigest(), 16)
        return random.Random(h)

    def fetch_page(self, page: str) -> Tuple[str, List[str], int]:
        key = canonicalize_page_key(page)
        rng = self._rng_for_key(key)

        # content length ~ 800-1200 words
        words = rng.choices(self.vocab, k=rng.randint(800, 1200))
        content = " ".join(words)

        # deterministic outlinks
        outlinks = []
        for _ in range(self.out_degree):
            j = rng.randrange(self.n_pages)
            outlinks.append(f"Page_{j}")
        outlinks = list(dict.fromkeys(outlinks))  # dedup

        return content, outlinks, 200


## 5) Cache-aware `WikipediaPage`

This wrapper:
- canonicalizes page keys
- checks the DB cache
- fetches via scraper only if missing
- writes `PageRecord` + `EdgeRecord`s (batched or direct)


In [ ]:
class WikipediaPage:
    def __init__(self, page: str, store: WikiSqliteStore, scraper, batch: Optional[CrawlBatchWriter] = None):
        self.store = store
        self.scraper = scraper
        self.key = canonicalize_page_key(page)
        self.url = wiki_url(self.key, self.store.base_url)
        self._content: Optional[str] = None
        self._outlinks: Optional[List[str]] = None  # set when freshly fetched (so crawling doesn't depend on DB flush)

        row = self.store.get_page(self.key)
        if row is not None and row["content"]:
            self._content = row["content"]
            self._outlinks = None  # read outlinks from DB on demand
        else:
            content, outlinks, status = self.scraper.fetch_page(self.key)
            self._content = content
            self._outlinks = outlinks

            now = utc_now_iso()
            page_rec = PageRecord(
                page=self.key,
                url=self.url,
                content=content,
                status_code=status,
                fetched_at=now,
            )
            edge_recs = [EdgeRecord(src=self.key, dst=dst, discovered_at=now) for dst in outlinks]

            if batch is not None:
                batch.add_page(page_rec)
                batch.add_edges(edge_recs)
            else:
                self.store.bulk_write([page_rec], edge_recs)

    def get_page_name(self) -> str:
        return self.key

    def get_content(self) -> str:
        return self._content or ""

    def get_hyperlinks(self, limit: Optional[int] = None) -> List[str]:
        # If newly fetched, return in-memory outlinks immediately (batch may not be flushed yet).
        if self._outlinks is not None:
            if limit is None:
                return list(self._outlinks)
            return list(self._outlinks)[: int(limit)]
        return self.store.neighbors(self.key, limit=limit)


## 6) A crawl builder that writes efficiently

`WikiNetworkBuilder` performs a bounded BFS crawl (depth + max connections).
It **does not** require building a full in-memory graph during the crawl.

For visualization/analysis, it can later extract a subgraph from the DB.


In [ ]:
class WikiNetworkBuilder:
    def __init__(self, start_page: str, store: WikiSqliteStore, scraper):
        self.store = store
        self.scraper = scraper
        self.start_key = canonicalize_page_key(start_page)

    def crawl(self, depth: int = 2, max_connections: int = 10, seed: int = 0,
              batch_pages: int = 50, batch_edges: int = 10_000) -> Dict[str, Any]:
        rng = random.Random(seed)
        visited = set()
        frontier = [self.start_key]
        n_fetched = 0

        with CrawlBatchWriter(self.store, batch_pages=batch_pages, batch_edges=batch_edges) as batch:
            for d in range(depth):
                next_frontier = []
                for key in frontier:
                    if key in visited:
                        continue
                    visited.add(key)

                    # Ensure cached (fetch if needed)
                    page = WikipediaPage(key, store=self.store, scraper=self.scraper, batch=batch)
                    n_fetched += 1

                    outlinks = page.get_hyperlinks()
                    if max_connections is not None and len(outlinks) > max_connections:
                        outlinks = rng.sample(outlinks, k=max_connections)

                    next_frontier.extend(outlinks)

                frontier = next_frontier

        return {
            "start": self.start_key,
            "depth": depth,
            "visited": len(visited),
            "fetched": n_fetched,
        }

    def extract_subgraph(self, center: str, max_nodes: int = 200) -> nx.DiGraph:
        """Extract a small subgraph around `center` for visualization."""
        center = canonicalize_page_key(center)
        g = nx.DiGraph()
        q = [center]
        seen = set()

        while q and len(seen) < max_nodes:
            u = q.pop(0)
            if u in seen:
                continue
            seen.add(u)
            g.add_node(u)
            nbrs = self.store.neighbors(u, limit=50)
            for v in nbrs:
                g.add_edge(u, v)
                if v not in seen and len(seen) + len(q) < max_nodes:
                    q.append(v)
        return g


## 7) Tests (fast, offline)

These are simple `assert`-based tests verifying:
- canonicalization
- storing pages/edges
- neighbor queries


In [ ]:
import tempfile
from pathlib import Path

def run_basic_tests():
    # canonicalization
    assert canonicalize_page_key("/wiki/Socrates") == "Socrates"
    assert canonicalize_page_key("Socrates") == "Socrates"
    assert canonicalize_page_key("Socrates#Life") == "Socrates"
    assert canonicalize_page_key("https://en.wikipedia.org/wiki/Python_(programming_language)#History") == "Python_(programming_language)"
    assert wiki_relpath("Socrates") == "/wiki/Socrates"

    # store roundtrip
    with tempfile.TemporaryDirectory() as td:
        db = str(Path(td) / "test.db")
        store = WikiSqliteStore(db_path=db)
        store.bulk_write(
            pages=[PageRecord(page="A", url=wiki_url("A"), content="hello", status_code=200, fetched_at="2026-01-01T00:00:00Z")],
            edges=[EdgeRecord(src="A", dst="B"), EdgeRecord(src="A", dst="C")]
        )
        row = store.get_page("A")
        assert row is not None
        assert row["content_sha256"] == sha256_text("hello")
        assert set(store.neighbors("A")) == {"B", "C"}
        stats = store.counts()
        assert stats["pages"] >= 3
        assert stats["edges"] == 2
        store.close()

run_basic_tests()
print("✅ basic tests passed")


## 8) Benchmarks: runtime + storage footprint

We empirically compare:

1) **Old schema** (single `wikipedia_pages` table with comma-separated hyperlinks)
2) **Refactored schema** (`pages` + `edges`, integer ids, batching)

Measured:
- Insert time
- Neighbor-query time
- SQLite file size


In [ ]:
class OldWikipediaStore:
    """Baseline matching the original notebook's storage approach."""
    def __init__(self, db_path: str):
        self.db_path = db_path
        self.conn = sqlite3.connect(db_path)
        self.conn.execute("""
        CREATE TABLE IF NOT EXISTS wikipedia_pages (
            id INTEGER PRIMARY KEY,
            page TEXT NOT NULL UNIQUE,
            content TEXT,
            hyperlinks TEXT
        );
        """)
        self.conn.commit()

    def close(self):
        self.conn.close()

    def insert_page(self, page: str, content: str, hyperlinks: List[str]):
        cur = self.conn.cursor()
        cur.execute(
            "INSERT OR REPLACE INTO wikipedia_pages(page, content, hyperlinks) VALUES(?,?,?)",
            (page, content, ",".join(hyperlinks))
        )
        self.conn.commit()

    def neighbors(self, page: str) -> List[str]:
        cur = self.conn.cursor()
        cur.execute("SELECT hyperlinks FROM wikipedia_pages WHERE page=?", (page,))
        row = cur.fetchone()
        if not row or not row[0]:
            return []
        return row[0].split(",")

    def db_component_sizes(self) -> Dict[str, int]:
        base = self.db_path
        parts = {"db": base, "wal": base + "-wal", "shm": base + "-shm"}
        sizes: Dict[str, int] = {}
        for name, path in parts.items():
            sizes[name] = os.path.getsize(path) if os.path.exists(path) else 0
        return sizes

    def disk_usage_bytes(self) -> int:
        return int(sum(self.db_component_sizes().values()))



def make_synth_dataset(n_pages: int = 3000, out_degree: int = 12, seed: int = 7):
    rng = random.Random(seed)
    keys = [f"Page_{i}" for i in range(n_pages)]
    vocab = [f"w{i}" for i in range(4000)]
    contents = {}
    links = {}

    for k in keys:
        contents[k] = " ".join(rng.choices(vocab, k=900))
        links[k] = [keys[rng.randrange(n_pages)] for _ in range(out_degree)]
    return keys, contents, links


def benchmark_old_vs_new(n_pages: int = 3000, out_degree: int = 12):
    import tempfile
    from pathlib import Path

    keys, contents, links = make_synth_dataset(n_pages=n_pages, out_degree=out_degree)

    with tempfile.TemporaryDirectory() as td:
        old_db = str(Path(td) / "old.db")
        new_db = str(Path(td) / "new.db")

        # --- OLD ---
        old = OldWikipediaStore(old_db)
        t0 = time.perf_counter()
        for k in keys:
            old.insert_page(k, contents[k], links[k])
        t_insert_old = time.perf_counter() - t0

        # neighbor query benchmark
        sample = random.sample(keys, k=min(200, len(keys)))
        t0 = time.perf_counter()
        for k in sample:
            _ = old.neighbors(k)
        t_neighbors_old = time.perf_counter() - t0
        old_size = old.disk_usage_bytes()
        old.close()

        # --- NEW ---
        store = WikiSqliteStore(new_db)
        t0 = time.perf_counter()

        with CrawlBatchWriter(store, batch_pages=100, batch_edges=50_000) as batch:
            for k in keys:
                batch.add_page(PageRecord(page=k, url=wiki_url(k), content=contents[k], status_code=200, fetched_at=utc_now_iso()))
                batch.add_edges([EdgeRecord(src=k, dst=dst) for dst in links[k]])

        t_insert_new = time.perf_counter() - t0

        # neighbor query benchmark
        t0 = time.perf_counter()
        for k in sample:
            _ = store.neighbors(k)
        t_neighbors_new = time.perf_counter() - t0
        store.conn.execute("PRAGMA wal_checkpoint(TRUNCATE);")
        new_size = store.disk_usage_bytes()
        counts = store.counts()
        store.close()

        df = pd.DataFrame([
            {"schema": "old_single_table", "insert_seconds": t_insert_old, "neighbors_seconds": t_neighbors_old, "db_size_mb": old_size/1e6},
            {"schema": "new_pages_edges_batched", "insert_seconds": t_insert_new, "neighbors_seconds": t_neighbors_new, "db_size_mb": new_size/1e6},
        ])
        meta = {"new_counts": counts}
        return df, meta

bench_df, bench_meta = benchmark_old_vs_new(n_pages=2000, out_degree=10)
display(bench_df)
bench_meta


### Batching vs no batching (same schema)

This isolates the performance gain from batching (transaction amortization).


In [ ]:
def benchmark_batching(n_pages: int = 2000, out_degree: int = 10):
    import tempfile
    from pathlib import Path

    keys, contents, links = make_synth_dataset(n_pages=n_pages, out_degree=out_degree)

    with tempfile.TemporaryDirectory() as td:
        db1 = str(Path(td) / "commit_each.db")
        db2 = str(Path(td) / "batched.db")

        # commit each write (worst case)
        store1 = WikiSqliteStore(db1)
        t0 = time.perf_counter()
        for k in keys:
            store1.bulk_write(
                pages=[PageRecord(page=k, url=wiki_url(k), content=contents[k], status_code=200, fetched_at=utc_now_iso())],
                edges=[EdgeRecord(src=k, dst=dst) for dst in links[k]]
            )
        t_commit_each = time.perf_counter() - t0
        store1.conn.execute("PRAGMA wal_checkpoint(TRUNCATE);")
        size1 = store1.disk_usage_bytes()
        store1.close()

        # batched
        store2 = WikiSqliteStore(db2)
        t0 = time.perf_counter()
        with CrawlBatchWriter(store2, batch_pages=100, batch_edges=50_000) as batch:
            for k in keys:
                batch.add_page(PageRecord(page=k, url=wiki_url(k), content=contents[k], status_code=200, fetched_at=utc_now_iso()))
                batch.add_edges([EdgeRecord(src=k, dst=dst) for dst in links[k]])
        t_batched = time.perf_counter() - t0
        store2.conn.execute("PRAGMA wal_checkpoint(TRUNCATE);")
        size2 = store2.disk_usage_bytes()
        store2.close()

        df = pd.DataFrame([
            {"mode": "commit_each_page", "seconds": t_commit_each, "db_size_mb": size1/1e6},
            {"mode": "batched_flush", "seconds": t_batched, "db_size_mb": size2/1e6},
        ])
        df["speedup_vs_commit_each"] = df["seconds"].iloc[0] / df["seconds"]
        return df

batch_df = benchmark_batching(n_pages=1500, out_degree=10)
display(batch_df)


## 9) Visualizations

We’ll crawl a small synthetic neighborhood (offline) and visualize:

- Static network plot (matplotlib + NetworkX)
- Interactive Plotly HTML export (higher fidelity)

This is directly useful later for explaining reasoning paths (highlight a path in the graph).


In [ ]:
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as td:
    db = str(Path(td) / "viz.db")
    store = WikiSqliteStore(db_path=db)
    scraper = FakeWikipediaScraper(n_pages=2000, out_degree=10, seed=42)

    builder = WikiNetworkBuilder(start_page="Page_0", store=store, scraper=scraper)
    crawl_stats = builder.crawl(depth=3, max_connections=6, seed=0, batch_pages=50, batch_edges=20_000)
    print("crawl_stats:", crawl_stats)
    print("store_counts:", store.counts(), "db_size_mb:", store.disk_usage_bytes()/1e6)

    g = builder.extract_subgraph(center="Page_0", max_nodes=120)

    # --- static plot ---
    plt.figure(figsize=(12, 8))
    pos = nx.spring_layout(g, seed=7, k=0.6)
    nx.draw(g, pos, node_size=70, alpha=0.8, with_labels=False, arrowsize=8)
    # highlight start
    nx.draw_networkx_nodes(g, pos, nodelist=["Page_0"], node_size=250)
    plt.title("Subgraph around Page_0 (static)")
    plt.show()

    # --- degree distribution ---
    degrees = [d for _, d in g.out_degree()]
    plt.figure(figsize=(8, 4))
    plt.hist(degrees, bins=20)
    plt.title("Out-degree distribution (subgraph)")
    plt.xlabel("out-degree")
    plt.ylabel("count")
    plt.show()

    # --- interactive Plotly export ---
    # Build edge traces
    edge_x = []
    edge_y = []
    for u, v in g.edges():
        x0, y0 = pos[u]
        x1, y1 = pos[v]
        edge_x += [x0, x1, None]
        edge_y += [y0, y1, None]

    edge_trace = go.Scatter(
        x=edge_x, y=edge_y,
        line=dict(width=1),
        hoverinfo='none',
        mode='lines'
    )

    # Node traces
    node_x = []
    node_y = []
    text = []
    for node in g.nodes():
        x, y = pos[node]
        node_x.append(x); node_y.append(y)
        text.append(node)

    node_trace = go.Scatter(
        x=node_x, y=node_y,
        mode='markers',
        hoverinfo='text',
        text=text,
        marker=dict(
            size=8,
            line=dict(width=0.5)
        )
    )

    fig = go.Figure(data=[edge_trace, node_trace])
    fig.update_layout(
        title="Interactive subgraph (Plotly)",
        showlegend=False,
        hovermode='closest',
        margin=dict(l=10, r=10, t=40, b=10),
        xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
        yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    )

    # html_path = "/mnt/data/siteHopper_graph_interactive.html"
    # pio.write_html(fig, file=html_path, auto_open=False, include_plotlyjs="cdn")
    # print("Saved interactive HTML to:", html_path)

    store.close()


## 10) Next steps (where your research direction plugs in)

Now that the store is graph-native, you can implement:

- bounded BFS / beam search / best-first over `store.neighbors(node)`
- semantic pruning: prune candidates by TF-IDF similarity, embeddings, or ontology constraints
- reasoning-path logging: record the path of expansions (`parent[node]=...`)
- explainability export: render the chosen path on the Plotly graph (highlighted edges)

If you want, I can add:
- a `bounded_best_first(query, start_key, ...)` implementation scaffold
- path provenance objects that store edge-level evidence/weights
- export as JSONL for analysis across many queries
